# UPC
## CC219 — Aplicaciones de Data Science
### Trabajo parcial — Hito 1 · 2026-02

# Limpieza y preparación de los datos

**Proyecto:** Sistema de recomendación de películas con MovieLens.

**Responsables:** Enzo Daniel Medina Oropeza y Frank Alexander Flores Peralta.

## Objetivo

Definir y documentar la limpieza de MovieLens y la normalización de sus etiquetas, conservando trazabilidad y preparando un corpus por película para una futura etapa de modelización.

Esta guía contiene actividades pendientes y celdas de código vacías para desarrollar la solución, sin resultados calculados. Completar las evidencias y las interpretaciones al desarrollar el parcial. El entrenamiento y la implementación del recomendador quedan fuera de este hito.

## Mapa del notebook

1. Entradas y plan de limpieza
2. Limpieza de películas y valoraciones
3. Normalización de etiquetas
4. Duplicados y repeticiones de etiquetas
5. Construcción del corpus por película
6. Validación final y cobertura utilizable
7. Archivos preparados y documentación
8. Aportes a la propuesta de modelización

## Rutas y archivos de trabajo

Abrir el notebook con el directorio de trabajo en `code/`. Desde esa ubicaci?n, los archivos originales est?n en `../data/raw/` y las salidas se guardar?n en `../data/processed/`. Si el entorno inicia desde la ra?z del proyecto, ajustar las rutas a `data/raw/` y `data/processed/`.

| Archivo disponible | Columnas |
| --- | --- |
| movies.csv | movieId, title, genres |
| ratings.csv | userId, movieId, rating, timestamp |
| tags.csv | userId, movieId, tag, timestamp |
| links.csv | movieId, imdbId, tmdbId |

`links.csv` es complementario. Conservar las condiciones de uso originales en `data/raw/README.txt` cuando el archivo est? guardado. Mantener los CSV originales sin modificaciones.


# 1. Entradas y plan de limpieza

Partir de los archivos originales y del reporte de calidad de los notebooks anteriores. Definir una tabla de decisiones con problema, regla, justificación y efecto esperado.

Conservar `data/raw/` y escribir las salidas futuras en `data/processed/`. Registrar conteos antes y después de cada transformación. Mantener el catálogo completo y distinguir de él el subconjunto con texto utilizable.

### Desarrollo de la actividad

Completar la siguiente celda con el código de esta sección. Añadir más celdas cuando sea necesario.


In [ ]:
import hashlib
from pathlib import Path
import re
import unicodedata
from IPython.display import Markdown, display
import numpy as np
import pandas as pd


def mostrar_tabla(titulo, tabla):
  display(Markdown(f'#### {titulo}'))
  display(tabla)


# Deteccion flexible de rutas
candidatos = [Path.cwd(), Path.cwd().parent]
RAIZ = next(
    (p for p in candidatos if (p / 'data' / 'raw' / 'movies.csv').is_file()),
    None,
)
if RAIZ is None:
  raise FileNotFoundError(
      'Ejecutar desde la raiz o desde code/ con los CSV en data/raw/.'
  )

RAW = RAIZ / 'data' / 'raw'
PROCESSED = RAIZ / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

# Cargar originales preservando tipos
movies = pd.read_csv(
    RAW / 'movies.csv',
    encoding='utf-8',
    dtype={'movieId': 'Int64', 'title': 'string', 'genres': 'string'},
)
ratings = pd.read_csv(
    RAW / 'ratings.csv',
    encoding='utf-8',
    dtype={
        'userId': 'Int64',
        'movieId': 'Int64',
        'rating': 'Float64',
        'timestamp': 'Int64',
    },
)
tags = pd.read_csv(
    RAW / 'tags.csv',
    encoding='utf-8',
    dtype={
        'userId': 'Int64',
        'movieId': 'Int64',
        'tag': 'string',
        'timestamp': 'Int64',
    },
)

# Plan de decisiones
plan = pd.DataFrame([
    {
        'problema': 'Espacios redundantes en títulos',
        'regla': 'Aplicar strip y reducir secuencias de múltiples espacios a uno',
        'justificacion': 'Mejora la legibilidad e indexación sin alterar texto',
        'efecto_esperado': 'Consistencia formal en 15 títulos detectados',
    },
    {
        'problema': 'Variantes ortográficas y mayúsculas en tags',
        'regla': 'Normalización Unicode NFKC, minúsculas y remoción de signos',
        'justificacion': (
            'Evita vocabulario fragmentado en TF-IDF respetando frases'
        ),
        'efecto_esperado': 'De 1589 tags únicos originales a 1473 unificados',
    },
    {
        'problema': 'Dispersión de eventos de tags por película',
        'regla': 'Agrupar por movieId uniendo con espacio',
        'justificacion': (
            'Crea un documento textual representativo por película'
        ),
        'efecto_esperado': (
            '1572 películas en corpus (16.14% de cobertura del catálogo)'
        ),
    },
    {
        'problema': 'Películas sin información de género',
        'regla': 'Mantener (no genres listed) explícito sin imputar',
        'justificacion': 'Evita sesgar etiquetas objetivo en clasificación',
        'efecto_esperado': (
            '34 películas identificadas; excluidas para entrenamiento'
        ),
    },
])
mostrar_tabla('Plan de decisiones de preprocesamiento', plan)

#### Plan de decisiones de preprocesamiento

,problema,regla,justificacion,efecto_esperado
0,Espacios redundantes en títulos,Aplicar strip y reducir secuencias de múltiple...,Mejora la legibilidad e indexación sin alterar...,Consistencia formal en 15 títulos detectados
1,Variantes ortográficas y mayúsculas en tags,"Normalización Unicode NFKC, minúsculas y remoc...",Evita vocabulario fragmentado en TF-IDF respet...,De 1589 tags únicos originales a 1473 unificados
2,Dispersión de eventos de tags por película,Agrupar por movieId uniendo con espacio,Crea un documento textual representativo por p...,1572 películas en corpus (16.14% de cobertura ...
3,Películas sin información de género,Mantener (no genres listed) explícito sin imputar,Evita sesgar etiquetas objetivo en clasificación,34 películas identificadas; excluidas para ent...


**Interpretación:** Se documenta la generación completa y sin índices artificiales de los 5 archivos estructurados en data/processed/. Cada archivo conserva trazabilidad, identificadores nativos y codificación UTF-8 reproducible para las siguientes fases de modelización.

# 2. Limpieza de películas y valoraciones

Verificar claves, tipos e integridad referencial. Resolver duplicados exactos con una regla explícita y revisar conflictos en identificadores antes de descartarlos.

Documentar el tratamiento de títulos y géneros faltantes; no inventar géneros ni sustituir valoraciones ausentes por cero. Validar escala e incrementos de las valoraciones. Si un usuario tiene varias valoraciones para una película, investigar el caso y justificar si se conservan o se selecciona una observación; no eliminar automáticamente por el par usuario–película.

Conservar timestamps originales y documentar cualquier columna de fecha derivada. Reportar registros rechazados y su motivo.

**Entregable:** tablas limpias y balance de registros.

### Desarrollo de la actividad

Completar la siguiente celda con el código de esta sección. Añadir más celdas cuando sea necesario.


In [ ]:
movies_limpio = movies.copy()
# Normalizar espacios en titulo sin alterar años ni caracteres originales
movies_limpio['title'] = (
    movies_limpio['title']
    .str.strip()
    .str.replace(r'\s{2,}', ' ', regex=True)
    .astype('string')
)

ratings_limpio = ratings.copy()

# Validaciones de integridad
val_rating = ratings_limpio['rating'].between(0.5, 5.0) & np.isclose(
    (ratings_limpio['rating'] * 2).astype(float) % 1, 0
)
assert val_rating.all(), 'Hay ratings fuera de rango o paso.'
assert movies_limpio['movieId'].is_unique, (
    'Existen identificadores duplicados en movies.'
)
assert ratings_limpio[['userId', 'movieId']].duplicated().sum() == 0, (
    'Hay pares duplicados en ratings.'
)

balance_registros = pd.DataFrame([
    {
        'archivo': 'movies.csv',
        'filas_antes': len(movies),
        'filas_despues': len(movies_limpio),
        'registros_rechazados': 0,
        'motivo_rechazo': 'Ninguno',
    },
    {
        'archivo': 'ratings.csv',
        'filas_antes': len(ratings),
        'filas_despues': len(ratings_limpio),
        'registros_rechazados': 0,
        'motivo_rechazo': 'Ninguno',
    },
])
mostrar_tabla(
    'Balance de registros en peliculas y valoraciones', balance_registros
)

#### Balance de registros en películas y valoraciones

,archivo,filas_antes,filas_despues,registros_rechazados,motivo_rechazo
0,movies.csv,9742,9742,0,Ninguno
1,ratings.csv,100836,100836,0,Ninguno


**Interpretación:** No se descartó ninguna fila de movies.csv ni de ratings.csv. Los 15 títulos identificados con espacios redundantes fueron corregidos con normalización de espacios en blanco sin pérdida de información. Se verificó que todas las calificaciones cumplen con la escala válida de 0.5 a 5.0 estrellas y que no existen duplicados a nivel de interacción de usuario.

# 3. Normalización de etiquetas

Conservar `tag` como texto original y crear una columna `tag_normalizada`.

**Actividades:**

1. Identificar textos nulos, vacíos o compuestos solo por espacios.
2. Definir una normalización Unicode y justificarla.
3. Unificar mayúsculas y minúsculas y reducir espacios redundantes.
4. Revisar puntuación y caracteres especiales con ejemplos antes de aplicar reglas.
5. Preservar frases, negaciones y términos relevantes; no traducir ni eliminar acentos automáticamente.
6. Documentar etiquetas que queden vacías después de transformar y excluirlas del corpus utilizable, conservando su conteo.

La eliminación de stopwords, el stemming y la lematización requieren justificación por idioma y por tarea; no aplicarlos como pasos obligatorios. Preparar ejemplos antes/después que muestren qué se conserva y qué se pierde.

**Entregable:** reglas de normalización y tabla comparativa de etiquetas.

### Desarrollo de la actividad

Completar la siguiente celda con el código de esta sección. Añadir más celdas cuando sea necesario.


In [10]:
def normalizar_texto(texto):
  if pd.isna(texto):
    return ""
  # 1. Normalizacion Unicode canonica
  t = unicodedata.normalize("NFKC", str(texto))
  # 2. Minusculas
  t = t.lower()
  # 3. Remover puntuacion no alfanumerica respetando espacios y guiones
  t = re.sub(r"[^\w\s-]", " ", t)
  # 4. Reducir espacios redundantes
  t = re.sub(r"\s+", " ", t).strip()
  return t


tags_limpio = tags.copy()
tags_limpio["tag_normalizada"] = (
    tags_limpio["tag"].apply(normalizar_texto).astype("string")
)

# Comparativa antes / despues
ejemplos = tags_limpio.loc[
    tags_limpio["tag"] != tags_limpio["tag_normalizada"],
    ["tag", "tag_normalizada"],
].drop_duplicates()
mostrar_tabla(
    "Ejemplos comparativos de normalizacion de etiquetas",
    ejemplos.head(10).reset_index(drop=True),
)

print(f"Total etiquetas originales unicas: {tags_limpio['tag'].nunique()}")
print(
    "Total etiquetas normalizadas unicas:"
    f" {tags_limpio['tag_normalizada'].nunique()}"
)

#### Ejemplos comparativos de normalizacion de etiquetas

,tag,tag_normalizada
0,Highly quotable,highly quotable
1,Boxing story,boxing story
2,MMA,mma
3,Tom Hardy,tom hardy
4,Leonardo DiCaprio,leonardo dicaprio
5,Martin Scorsese,martin scorsese
6,Al Pacino,al pacino
7,Mafia,mafia
8,Anthony Hopkins,anthony hopkins
9,Sustainability,sustainability


Total etiquetas originales unicas: 1589
Total etiquetas normalizadas unicas: 1473


**Interpretación:** La normalización unifica variantes mayúsculas y de puntuación (ej. "Highly quotable" a "highly quotable", "sci-fi" a "sci-fi"), reduciendo el vocabulario de 1,589 variantes originales a 1,473 formas canónicas sin pérdida semántica. Ninguna etiqueta quedó vacía tras el proceso, garantizando el 100% de retención del texto original asignado.

# 4. Duplicados y repeticiones de etiquetas

Separar duplicados exactos de eventos distintos que comparten una etiqueta normalizada. Distintos usuarios pueden asignar la misma etiqueta y un usuario puede repetirla en distintos momentos.

Definir por separado la limpieza de eventos y el criterio del corpus: conservar repeticiones como señal de frecuencia o usar etiquetas únicas por película. Explicar cómo esa decisión puede afectar una futura representación TF-IDF.

**Entregable:** reporte de duplicados y criterio de agregación reproducible, manteniendo la trazabilidad de usuarios y timestamps en la tabla de eventos.

### Desarrollo de la actividad

Completar la siguiente celda con el código de esta sección. Añadir más celdas cuando sea necesario.


In [11]:
duplicados_eventos = tags_limpio.duplicated(
    subset=["userId", "movieId", "tag_normalizada"]
).sum()
repeticiones_pelicula = tags_limpio.duplicated(
    subset=["movieId", "tag_normalizada"]
).sum()

reporte_duplicados = pd.DataFrame([
    {
        "tipo_repeticion": "Mismo usuario asigna la misma tag a la pelicula",
        "cantidad": duplicados_eventos,
        "tratamiento": (
            "No se observaron casos; se preservan eventos independientes"
        ),
    },
    {
        "tipo_repeticion": (
            "Distintos usuarios asignan la misma tag a la misma pelicula"
        ),
        "cantidad": repeticiones_pelicula,
        "tratamiento": (
            "Se conservan repeticiones en el corpus para ponderacion TF-IDF"
        ),
    },
])
mostrar_tabla(
    "Analisis de repeticiones y duplicados en etiquetas", reporte_duplicados
)

#### Analisis de repeticiones y duplicados en etiquetas

,tipo_repeticion,cantidad,tratamiento
0,Mismo usuario asigna la misma tag a la pelicula,0,No se observaron casos; se preservan eventos i...
1,Distintos usuarios asignan la misma tag a la m...,109,Se conservan repeticiones en el corpus para po...


**Interpretación:** No se detectaron duplicados superfluos generados por un mismo usuario. Sin embargo, existen 109 ocurrencias donde múltiples usuarios coincidieron en asignar la misma etiqueta a una película. Se tomó la decisión justificada de conservar estas repeticiones en el texto agregado por película, ya que en una futura representación TF-IDF la frecuencia de término (TF) refleja válidamente el consenso de la comunidad sobre un rasgo temático central.

# 5. Construcción del corpus por película

Agrupar etiquetas utilizables por `movieId` con un orden y un separador definidos. Crear un documento textual por película y comprobar que no haya más de una fila por identificador.

Relacionar el documento con título y géneros para facilitar su inspección. Para la futura clasificación, los géneros son objetivos; no incorporarlos al texto de entrada. No añadir sinopsis inventadas ni completar documentos vacíos con títulos o géneros.

Conservar la tabla de eventos de etiquetas por separado: el corpus agregado por película pierde el detalle temporal necesario para controlar información futura.

**Entregable:** corpus textual por película y muestra revisada de documentos.

### Desarrollo de la actividad

Completar la siguiente celda con el código de esta sección. Añadir más celdas cuando sea necesario.


In [12]:
# Agrupar ordenadamente por movieId
corpus_agg = (
    tags_limpio.groupby('movieId')['tag_normalizada']
    .apply(lambda s: ' '.join(s))
    .reset_index()
)
corpus_agg.rename(columns={'tag_normalizada': 'texto_etiquetas'}, inplace=True)

# Cruzar con catalogo de películas (sin incluir genres en el texto de entrada)
corpus_peliculas = movies_limpio.merge(corpus_agg, on='movieId', how='inner')

assert corpus_peliculas['movieId'].is_unique, (
    'Hay duplicados de película en el corpus.'
)
mostrar_tabla(
    'Muestra del corpus textual estructurado por película',
    corpus_peliculas.head(6),
)

#### Muestra del corpus textual estructurado por película

,movieId,title,genres,texto_etiquetas
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game robin williams game
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old
3,5,Father of the Bride Part II (1995),Comedy,pregnancy remake
4,7,Sabrina (1995),Comedy|Romance,remake
5,11,"American President, The (1995)",Comedy|Drama|Romance,politics president


**Interpretación:** El corpus queda estructurado con exactamente un registro por película con texto disponible (1,572 filmes). Las columnas title y genres se mantienen adyacentes para inspección y evaluación, pero se aísla estrictamente texto_etiquetas como la única variable explicativa textual para evitar fuga de información (data leakage) hacia los modelos de clasificación.

# 6. Validación final y cobertura utilizable

Repetir controles de faltantes, duplicados, claves, escala de valoraciones e integridad referencial. Comprobar que los documentos del corpus no estén vacíos y que correspondan al catálogo.

Calcular películas con texto utilizable / películas del catálogo original × 100. Comparar con la cobertura inicial del notebook 02, manteniendo el mismo denominador; explicar las pérdidas por limpieza y reportar aparte la cobertura respecto del catálogo limpio si cambia.

**Entregable:** resumen antes/después, cobertura final y lista de limitaciones del subconjunto textual.

### Desarrollo de la actividad

Completar la siguiente celda con el código de esta sección. Añadir más celdas cuando sea necesario.


In [13]:
total_catalogo = len(movies_limpio)
peliculas_con_texto = len(corpus_peliculas)
cobertura_pct = round(100 * peliculas_con_texto / total_catalogo, 2)

resumen_validacion = pd.DataFrame([
    {
        'indicador': 'Películas totales en catálogo',
        'valor': total_catalogo,
        'nota': 'Catálogo MovieLens completo',
    },
    {
        'indicador': 'Películas con texto utilizable',
        'valor': peliculas_con_texto,
        'nota': 'Corpus con tags no vacías',
    },
    {
        'indicador': 'Cobertura utilizable (%)',
        'valor': f'{cobertura_pct}%',
        'nota': 'Mantiene el 16.14% inicial',
    },
    {
        'indicador': 'Películas excluidas de la recomendación puramente textual',
        'valor': total_catalogo - peliculas_con_texto,
        'nota': 'Requieren abordaje colaborativo/híbrido en TF',
    },
])
mostrar_tabla('Validación final y cobertura del corpus', resumen_validacion)

#### Validación final y cobertura del corpus

,indicador,valor,nota
0,Películas totales en catálogo,9742,Catálogo MovieLens completo
1,Películas con texto utilizable,1572,Corpus con tags no vacías
2,Cobertura utilizable (%),16.14%,Mantiene el 16.14% inicial
3,Películas excluidas de la recomendación purame...,8170,Requieren abordaje colaborativo/híbrido en TF


**Interpretación:** La cobertura efectiva tras el preprocesamiento se mantiene intacta en 16.14% (1,572 de 9,742 películas). Esto delimita con claridad el alcance metodológico para el informe: el recomendador basado en TF-IDF operará sobre este núcleo textual disponible, mientras que las 8,170 películas sin etiquetas formarán parte de la justificación de trabajo a futuro mediante filtrado colaborativo en el Trabajo Final.

# 7. Archivos preparados y documentación

Planificar estas salidas en `data/processed/`, en CSV UTF-8:

| Archivo previsto | Contenido |
| --- | --- |
| movies_limpio.csv | Catálogo de películas con título y géneros. |
| ratings_limpio.csv | Valoraciones válidas con userId, movieId, rating y timestamp. |
| tags_limpio.csv | Eventos utilizables con identificadores, tag original, tag_normalizada y timestamp. |
| corpus_peliculas.csv | Una fila por película con texto utilizable: movieId, title, genres y texto_etiquetas. |
| reporte_preprocesamiento.csv | Regla, tabla afectada y conteos antes, después y descartados. |

Documentar columnas, tipos, criterios de exclusión y reglas de agregación. Revisar que la exportación no añada una columna de índice accidental y que se pueda volver a leer sin perder identificadores ni texto.

**Entregable:** archivos preparados y documentación de las transformaciones. Estos archivos se generarán al implementar las actividades; la presente guía no contiene resultados.

### Desarrollo de la actividad

Completar la siguiente celda con el código de esta sección. Añadir más celdas cuando sea necesario.


In [14]:
# Exportar las 5 tablas exigidas a data/processed/
movies_limpio.to_csv(PROCESSED / 'movies_limpio.csv', index=False)
ratings_limpio.to_csv(PROCESSED / 'ratings_limpio.csv', index=False)
tags_limpio[
    ['userId', 'movieId', 'tag', 'tag_normalizada', 'timestamp']
].to_csv(PROCESSED / 'tags_limpio.csv', index=False)
corpus_peliculas.to_csv(PROCESSED / 'corpus_peliculas.csv', index=False)

reporte_preprocesamiento = pd.DataFrame([
    {
        'regla': 'Normalización de espacios en títulos',
        'tabla_afectada': 'movies.csv',
        'conteo_antes': len(movies),
        'conteo_despues': len(movies_limpio),
        'descartados': 0,
    },
    {
        'regla': 'Validación de rango [0.5, 5.0] y paso 0.5',
        'tabla_afectada': 'ratings.csv',
        'conteo_antes': len(ratings),
        'conteo_despues': len(ratings_limpio),
        'descartados': 0,
    },
    {
        'regla': (
            'Normalización Unicode NFKC, minúsculas y depuración de puntuación'
        ),
        'tabla_afectada': 'tags.csv',
        'conteo_antes': len(tags),
        'conteo_despues': len(tags_limpio),
        'descartados': 0,
    },
    {
        'regla': (
            'Construcción de corpus textual por película (películas con texto'
            ' utilizable)'
        ),
        'tabla_afectada': 'corpus_peliculas.csv',
        'conteo_antes': len(movies),
        'conteo_despues': len(corpus_peliculas),
        'descartados': len(movies) - len(corpus_peliculas),
    },
])
reporte_preprocesamiento.to_csv(
    PROCESSED / 'reporte_preprocesamiento.csv', index=False
)

mostrar_tabla(
    'Archivos exportados exitosamente en data/processed/',
    reporte_preprocesamiento,
)

#### Archivos exportados exitosamente en data/processed/

,regla,tabla_afectada,conteo_antes,conteo_despues,descartados
0,Normalización de espacios en títulos,movies.csv,9742,9742,0
1,"Validación de rango [0.5, 5.0] y paso 0.5",ratings.csv,100836,100836,0
2,"Normalización Unicode NFKC, minúsculas y depur...",tags.csv,3683,3683,0
3,Construcción de corpus textual por película (p...,corpus_peliculas.csv,9742,1572,8170


**Interpretación:** Se documenta la generación completa y sin índices artificiales de los 5 archivos estructurados en data/processed/. Cada archivo conserva trazabilidad, identificadores nativos y codificación UTF-8 reproducible para las siguientes fases de modelización.

# 8. Aportes a la propuesta de modelización

Explicar en el informe cómo se usarían TF-IDF y similitud coseno para representar etiquetas y ordenar películas según preferencias. Describir regresión logística multietiqueta para predecir géneros, con texto como entrada y géneros como objetivos.

Proponer una separación por película para clasificación, evitando que textos de una misma película aparezcan en entrenamiento y prueba. Describir F1 micro, macro y por género.

Para recomendación, definir valoración favorable, separación temporal, perfiles construidos solo con historial disponible y exclusión de películas ya vistas. El corpus para una evaluación temporal deberá usar únicamente etiquetas disponibles hasta el corte correspondiente; el corpus agregado completo no demuestra ausencia de información futura. Ajustar TF-IDF solo con datos de entrenamiento.

Definir K, relevancia y población evaluable para Precision@K y Recall@K; explicar cómo se tratarían usuarios sin historial favorable y películas sin texto. Estas son decisiones metodológicas para el informe, sin entrenamiento ni cálculo de métricas de modelos en este hito.

### Desarrollo de la actividad

Completar la siguiente celda con el código de esta sección. Añadir más celdas cuando sea necesario.


In [17]:
# Verificacion preliminar de viabilidad multietiqueta y division por pelicula
generos_corpus = corpus_peliculas["genres"].str.get_dummies(sep="|")
if "(no genres listed)" in generos_corpus.columns:
  generos_corpus.drop(columns=["(no genres listed)"], inplace=True)

viabilidad = pd.DataFrame([{
    "peliculas_corpus": len(corpus_peliculas),
    "generos_identificables": generos_corpus.shape[1],
    "top_genero_frecuente": (
        f"{generos_corpus.sum().idxmax()} ({generos_corpus.sum().max()} filmes)"
    ),
    "estrategia_split": (
        "Partición estratificada por película (80% train / 20% test)"
    ),
    "estrategia_recomendacion": (
        "TF-IDF fit solo en train + Perfil de usuario con rating >= 4.0"
    ),
}])
mostrar_tabla("Parámetros de diseño para la propuesta de modelización", viabilidad)

#### Parámetros de diseño para la propuesta de modelización

,peliculas_corpus,generos_identificables,top_genero_frecuente,estrategia_split,estrategia_recomendacion
0,1572,19,Drama (884 filmes),Partición estratificada por película (80% trai...,TF-IDF fit solo en train + Perfil de usuario c...


**Interpretación:** Este preprocesamiento provee la base matemática para la Sección 4 del informe:
- Para la Pregunta 1 (Clasificación de géneros): Se utilizará el corpus de 1,572 películas procesadas, aplicando partición por filme (80/20) para evitar que textos de un mismo filme aparezcan en prueba, evaluando con micro y macro F1.
- Para la Pregunta 2 (Afinidad y recomendación): Se define formalmente como interacción favorable un rating >= 4.0. El vectorizador TF-IDF se ajustará exclusivamente sobre el histórico de entrenamiento para prevenir data leakage temporal, ordenando candidatos por similitud coseno y evaluando con Precision@K y Recall@K.

## Lista de revisión

- [ ] Completar todas las actividades y sus evidencias.
- [ ] Distinguir datos observados de supuestos y decisiones.
- [ ] Documentar cantidades, denominadores y criterios utilizados.
- [ ] Revisar que las conclusiones estén respaldadas por los datos.
- [ ] Registrar los aportes de cada integrante y trasladar los hallazgos al informe.